# 5.4 通信优化 (Communication Optimization)

分布式训练中，通信往往是瓶颈。通信优化技术旨在减少通信开销，提高训练效率。

本节涵盖：
- 通信与计算重叠
- 梯度压缩
- Ring Attention

## 1. 通信与计算重叠

**目的**：隐藏通信延迟

**基本原理**：在进行当前层的梯度通信时，同时计算下一层的梯度，使通信时间被计算时间覆盖，减少总训练时间。

**实现方式**：
- 使用异步通信操作（如AllReduce的异步版本）
- 将梯度计算和通信分为不同流（CUDA stream）
- 在反向传播中，计算完一层梯度后立即启动通信，同时计算下一层

**效率提升**：
- 无重叠：总时间 = 计算时间 + 通信时间
- 有重叠：总时间 ≈ max(计算时间, 通信时间)

In [ ]:
import torch
import time

torch.manual_seed(42)

n_layers = 8
compute_time_per_layer = 10
comm_time_per_layer = 6

sequential_time = n_layers * (compute_time_per_layer + comm_time_per_layer)
overlapped_time = n_layers * compute_time_per_layer + comm_time_per_layer

print('=== Communication-Computation Overlap ===')
print(f'Layers: {n_layers}')
print(f'Compute per layer: {compute_time_per_layer}ms')
print(f'Communication per layer: {comm_time_per_layer}ms')
print(f'\nSequential (no overlap):')
print(f'  Total = {n_layers} x ({compute_time_per_layer} + {comm_time_per_layer}) = {sequential_time}ms')
print(f'\nOverlapped:')
print(f'  Total = {n_layers} x {compute_time_per_layer} + {comm_time_per_layer} = {overlapped_time}ms')
print(f'  Speedup: {sequential_time/overlapped_time:.2f}x')

print(f'\nTimeline comparison:')
print(f'  Sequential: [C1][C1_comm][C2][C2_comm]...[C8][C8_comm]')
print(f'  Overlapped: [C1][C2+C1_comm][C3+C2_comm]...[C8+C7_comm][C8_comm]')
print(f'\nKey: Overlapping hides communication latency behind computation.')
print(f'The more computation-dominant, the more effective the overlap.')

## 2. 梯度压缩

**目的**：减少通信数据量

**基本原理**：使用梯度量化或稀疏化方法减少通信量，适用于低带宽网络。

**主要方法**：
- **梯度量化**：将FP32梯度量化为8-bit或更低位，减少75%+通信量
- **梯度稀疏化**：仅传输绝对值最大的部分梯度（如Top-K稀疏化），其余本地累积
- **PowerSGD**：使用低秩近似压缩梯度

**精度影响**：梯度压缩会引入误差，但通常对训练收敛影响很小

In [ ]:
import torch

torch.manual_seed(42)

grad = torch.randn(1024, 1024)

def quantize_8bit(tensor):
    scale = tensor.abs().max() / 127.0
    quantized = torch.round(tensor / scale).clamp(-128, 127).to(torch.int8)
    return quantized, scale

def topk_sparsify(tensor, k_ratio=0.01):
    k = max(1, int(tensor.numel() * k_ratio))
    values, indices = tensor.abs().flatten().topk(k)
    sparse = torch.zeros_like(tensor).flatten()
    sparse[indices] = tensor.flatten()[indices]
    return sparse.reshape(tensor.shape), indices

grad_q, scale_q = quantize_8bit(grad)
grad_dq = grad_q.float() * scale_q
grad_sparse, sparse_idx = topk_sparsify(grad, k_ratio=0.01)

print('=== Gradient Compression ===')
print(f'Gradient shape: {grad.shape}')
print(f'Original size: {grad.numel() * 4 / 1e6:.2f} MB (FP32)')

print(f'\n8-bit Quantization:')
print(f'  Compressed size: {grad_q.numel() / 1e6:.2f} MB (4x reduction)')
q_error = (grad - grad_dq).norm() / grad.norm()
print(f'  Relative error: {q_error:.4f} ({q_error:.2%})')

print(f'\nTop-1% Sparsification:')
print(f'  Non-zero elements: {len(sparse_idx)} / {grad.numel()} = {len(sparse_idx)/grad.numel():.2%}')
print(f'  Communication: ~{len(sparse_idx) * 8 / 1e6:.2f} MB (values + indices)')
s_error = (grad - grad_sparse).norm() / grad.norm()
print(f'  Relative error: {s_error:.4f} (99% of gradient dropped!)')
print(f'\nKey: Gradient compression trades slight accuracy for major communication savings.')
print(f'In practice, 8-bit quantization has minimal impact on convergence.')

## 3. Ring Attention

**目的**：支持超长序列的分布式训练

**基本原理**：将序列沿维度切分到多个设备，使用环形通信传递KV块，使每台设备仅需存储1/N的KV，理论上支持无限长序列训练。

**Ring Attention工作流程**：
1. 每台设备持有一段Q和初始KV块
2. 计算当前KV块与Q的注意力
3. 将KV块传递给下一台设备，接收上一台设备的KV块
4. 重复直到所有KV块都被处理
5. 每台设备得到完整的注意力输出

**优势**：
- 序列长度不再受单GPU显存限制
- 通信与计算可以重叠
- 理论上支持无限长序列

In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(42)

B, H, D = 1, 4, 32
seq_len = 64
n_devices = 4
chunk_len = seq_len // n_devices

Q = torch.randn(B, H, seq_len, D)
K = torch.randn(B, H, seq_len, D)
V = torch.randn(B, H, seq_len, D)

attn_full = torch.softmax(Q @ K.transpose(-2, -1) / math.sqrt(D), dim=-1) @ V

output_ring = torch.zeros_like(Q)
for device_id in range(n_devices):
    q_local = Q[:, :, device_id*chunk_len:(device_id+1)*chunk_len, :]
    out_accum = torch.zeros(B, H, chunk_len, D)
    weight_accum = torch.zeros(B, H, chunk_len, 1)
    for step in range(n_devices):
        kv_device = (device_id + step) % n_devices
        k_block = K[:, :, kv_device*chunk_len:(kv_device+1)*chunk_len, :]
        v_block = V[:, :, kv_device*chunk_len:(kv_device+1)*chunk_len, :]
        scores = q_local @ k_block.transpose(-2, -1) / math.sqrt(D)
        max_score = scores.max(dim=-1, keepdim=True).values
        exp_scores = torch.exp(scores - max_score)
        out_accum += exp_scores @ v_block
        weight_accum += exp_scores.sum(dim=-1, keepdim=True)
    output_ring[:, :, device_id*chunk_len:(device_id+1)*chunk_len, :] = out_accum / weight_accum

print('=== Ring Attention ===')
print(f'Sequence length: {seq_len}, Devices: {n_devices}')
print(f'Per-device sequence chunk: {chunk_len}')
print(f'\nResults match standard attention: {torch.allclose(attn_full, output_ring, atol=1e-4)}')
print(f'Max difference: {(attn_full - output_ring).abs().max():.6f}')

full_kv_mem = B * H * seq_len * D * 2 * 4
ring_kv_mem = full_kv_mem / n_devices
print(f'\nKV Cache memory per device:')
print(f'  Standard: {full_kv_mem/1e6:.1f} MB')
print(f'  Ring Attention: {ring_kv_mem/1e6:.1f} MB ({1/n_devices:.0%} of standard)')
print(f'\nKey: Ring Attention distributes KV across devices in a ring pattern.')
print(f'Each device only stores 1/N of KV, enabling very long sequence training.')